# SAE Training — CLIP / SigLIP1 / SigLIP2
Trains Sparse Autoencoders on frozen ViT activations (hook_resid_post) for all 12 layers.

Run the first three cells to verify the Colab runtime before starting training.

## 1 — Runtime check

In [3]:
!nvidia-smi
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

: 

## 2 — Install dependencies

> **Note:** ViT-Prisma is installed from our fork which adds SigLIP1/2 support.
> Replace the URL below with your fork once you push the SigLIP changes.

In [4]:
import os, glob

FOLDER_ID = '16axcxIfAwW0ebpc_0NuJSeSy_RDY_zqt'
SAVE_DIR  = '/content/saes'
DATA_DIR  = '/content/imagenet_val'
PARQUET_GLOB = '/content/imagenet_val/data/*.parquet'

# force a clean re-download (previous run was partial)
# !rm -rf /content/imagenet_val
!pip install -q gdown
!gdown --folder "https://drive.google.com/drive/folders/{FOLDER_ID}" -O /content --quiet

print('contents of /content/imagenet_val:')
!ls -R /content/imagenet_val | head -40
print(f'\n{len(glob.glob(PARQUET_GLOB))} parquet files found')

contents of /content/imagenet_val:
/content/imagenet_val:
data

/content/imagenet_val/data:
validation-00000-of-00014.parquet
validation-00001-of-00014.parquet
validation-00001-of-00014.parquetazjjm3_u.part
validation-00002-of-00014.parquet
validation-00003-of-00014.parquet
validation-00004-of-00014.parquet
validation-00005-of-00014.parquet
validation-00006-of-00014.parquet
validation-00007-of-00014.parquet
validation-00008-of-00014.parquet
validation-00009-of-00014.parquet
validation-00010-of-00014.parquet
validation-00011-of-00014.parquet
validation-00012-of-00014.parquet
validation-00013-of-00014.parquet

14 parquet files found


In [5]:
!pip install -q transformers==4.44.2 einops timm datasets huggingface_hub tqdm
!pip install -q git+https://github.com/asharalam11/ViT-Prisma.git@add_siglip2

  Preparing metadata (setup.py) ... done


## 3 — Verify model loading and activation caching

In [6]:
import torch
from vit_prisma.models.model_loader import load_hooked_model

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)

# Crux test: does run_with_cache OOM on T4 with SigLIP's 211 hook points?
model = load_hooked_model('google/siglip-base-patch16-224', device=device)
model = model.to(device)   # ensure all weights are on GPU
model.eval()

# Only keep the hook points we train SAEs on (resid_post + mlp_out, all 12 layers)
wanted = {f'blocks.{l}.{h}' for l in range(12)
          for h in ['hook_resid_post', 'hook_mlp_out']}
names_filter = lambda n: n in wanted

x = torch.randn(8, 3, 224, 224, device=device)  # batch of 8
torch.cuda.reset_peak_memory_stats()
with torch.no_grad():
    _, cache = model.run_with_cache(x, names_filter=names_filter)

print(f'Cached {len(cache)} hook points')
print('resid_post[11]:', cache['blocks.11.hook_resid_post'].shape)
print(f'Peak VRAM: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB')

del model, cache, x
torch.cuda.empty_cache()
print('OK')

/usr/local/lib/python3.12/dist-packages/kaleido/_sync_server.py:11: UserWarning:




This means that static image generation (e.g. `fig.write_image()`) will not work.

Please upgrade Plotly to version 6.1.1 or greater, or downgrade Kaleido to version 0.2.1.

You can however, use the Kaleido API directly which will work with your plotly version. `kaleido.write_fig(...)`, for example. Please see the kaleido documentation.




Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning:


Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).



ln_pre not set
Cached 24 hook points
resid_post[11]: torch.Size([8, 196, 768])
Peak VRAM: 0.56 GB
OK


---
## 4 — SAE Training

Cells below run after the runtime check passes.

In [7]:
# Config — edit before running
MODEL_ID   = 'google/siglip-base-patch16-224'  # or siglip2, clip
HOOK_POINT = 'hook_resid_post'                 # or hook_mlp_out
LAYERS     = list(range(12))                   # all 12
N_PATCHES  = 196                               # 14x14, no CLS for SigLIP

SAE_DIM    = 768 * 4   # expansion factor 4x
L1_COEFF   = 1e-3
LR         = 1e-4
BATCH_SIZE = 64
N_STEPS    = 10_000    # adjust based on dataset size

print('Config OK')

Config OK


In [8]:
import torch
import torch.nn as nn

class SAE(nn.Module):
    def __init__(self, d_in, d_sae):
        super().__init__()
        self.W_enc = nn.Linear(d_in, d_sae, bias=True)
        self.W_dec = nn.Linear(d_sae, d_in, bias=True)
        self.relu  = nn.ReLU()
        # decoder columns normalised to unit norm
        with torch.no_grad():
            self.W_dec.weight.data = nn.functional.normalize(
                self.W_dec.weight.data, dim=0)

    def forward(self, x):
        f = self.relu(self.W_enc(x))
        x_hat = self.W_dec(f)
        return x_hat, f

    def normalise_decoder(self):
        with torch.no_grad():
            self.W_dec.weight.data = nn.functional.normalize(
                self.W_dec.weight.data, dim=0)

print('SAE class defined')

SAE class defined


In [9]:
from datasets import load_dataset
from torchvision import transforms

ds = load_dataset('parquet', data_files=PARQUET_GLOB, split='train')
print(f'Loaded {len(ds)} images')

preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

def collate(batch):
    imgs = torch.stack([preprocess(b['image'].convert('RGB')) for b in batch])
    return imgs

loader = torch.utils.data.DataLoader(ds, batch_size=BATCH_SIZE, collate_fn=collate, num_workers=2)
print(f'{len(loader)} batches of {BATCH_SIZE}')

Loaded 50000 images
782 batches of 64


In [10]:
from vit_prisma.sae import VisionModelSAERunnerConfig, VisionSAETrainer
from torchvision import transforms

  # The activations store expects each item as (image_tensor, label).
  # Our parquet HF dataset yields {'image': PIL, 'label': int}, so wrap it.
_preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

class ImageTupleDataset(torch.utils.data.Dataset):
    def __init__(self, hf_ds):
        self.ds = hf_ds
    def __len__(self):
        return len(self.ds)   
    def __getitem__(self, i):
        row = self.ds[i]
        return _preprocess(row['image'].convert('RGB')), row['label']

print('Dataset wrapper ready')

Dataset wrapper ready


In [11]:
from datasets import load_dataset
from vit_prisma.models.model_loader import load_hooked_model

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Load parquet and split off a small eval set (trainer requires eval_dataset)
full = load_dataset('parquet', data_files=PARQUET_GLOB, split='train')
split = full.train_test_split(test_size=0.02, seed=42)
train_ds = ImageTupleDataset(split['train'])
eval_ds  = ImageTupleDataset(split['test'])
print(f'train: {len(train_ds)}  eval: {len(eval_ds)}')

# Load the frozen vision model once, reused for every layer's SAE
model = load_hooked_model(MODEL_ID, device=device).to(device)
model.eval()
print('Model loaded on', device)

train: 49000  eval: 1000
ln_pre not set
Model loaded on cuda


In [12]:
import os

SMOKE_TEST = True   # True = ~5K images (~1 min); False = full 50K pass

def make_cfg(layer):
    return VisionModelSAERunnerConfig(
        model_name=MODEL_ID,
        model_class_name='HookedViT',
        hook_point_layer=layer,
        layer_subtype=HOOK_POINT,        # e.g. 'hook_resid_post'
        d_in=768,
        expansion_factor=4,
        context_size=196,                # 14x14 patches, no CLS for SigLIP
        image_size=224,
        activation_fn_str='relu',
        num_epochs=0.004 if SMOKE_TEST else 0.04,  # caps training length
        l1_coefficient=L1_COEFF,
        lr=LR,
        train_batch_size=4096,
        n_checkpoints=0,
        log_to_wandb=False,
        checkpoint_path=SAVE_DIR,
        verbose=True,
        _device=device,
    )

# Smoke test: train a single layer first to confirm the pipeline runs end to end
LAYER = 8
cfg = make_cfg(LAYER)
trainer = VisionSAETrainer(cfg, model, train_ds, eval_dataset=eval_ds)
sae = trainer.run()

os.makedirs(SAVE_DIR, exist_ok=True)
save_path = f'{SAVE_DIR}/sae_{MODEL_ID.split("/")[-1]}_{HOOK_POINT}_layer{LAYER}.pt'
torch.save(sae.state_dict(), save_path)
print('Saved to', save_path)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning:

This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning:

This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.



Not saving checkpoints so skipping creating checkpoint directory
Configuration:
  model_class_name: HookedViT
  model_name: google/siglip-base-patch16-224
  vit_model_cfg: None
  model_path: None
  hook_point_layer: 8
  layer_subtype: hook_resid_post
  hook_point_head_index: None
  context_size: 196
  use_cached_activations: False
  use_patches_only: False
  cached_activations_path: activations/_network_scratch_s_sonia.joseph_datasets_kaggle_datasets/google_siglip-base-patch16-224/blocks.8.hook_resid_post
  image_size: 224
  architecture: standard
  b_dec_init_method: geometric_median
  expansion_factor: 4
  from_pretrained_path: None
  is_transcoder: False
  transcoder_with_skip_connection: True
  out_hook_point_layer: 9
  layer_out_subtype: hook_mlp_out
  d_out: 768
  _device: cuda
  seed: 42
  _dtype: float32
  d_in: 768
  activation_fn_str: relu
  activation_fn_kwargs: {}
  cls_token_only: False
  max_grad_norm: 1.0
  initialization_method: independent
  normalize_activations: laye

Objective value: 1366117.1250:   2%|▏         | 3/200 [00:00<00:19, 10.22it/s]


Starting training


Training SAE: Loss: 0.2302, MSE Loss: 0.0186, L1 Loss: 0.2115, L0: 829.5808: : 1019904it [02:08, 7953.34it/s]                           

Final checkpoint saved at 1019904 tokens


Saved to /content/saes/sae_siglip-base-patch16-224_hook_resid_post_layer8.pt


In [14]:
from google.colab import auth 
auth.authenticate_user()
import google.auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload

FOLDER_ID = '16axcxIfAwW0ebpc_0NuJSeSy_RDY_zqt'
_creds, _ = google.auth.default()
_drive = build('drive', 'v3', credentials=_creds)

def upload_to_drive(path):
    meta  = {'name': os.path.basename(path), 'parents': [FOLDER_ID]}
    media = MediaFileUpload(path, resumable=True)
    f = _drive.files().create(body=meta, media_body=media, fields='id').execute()
    print('Uploaded', path, '->', f['id'])

In [15]:
 upload_to_drive('/content/saes/sae_siglip-base-patch16-224_hook_resid_post_layer8.pt')

Uploaded /content/saes/sae_siglip-base-patch16-224_hook_resid_post_layer8.pt -> 1axlyAajD4nHznVrdmQndWYu88zLWulR5


In [16]:
def make_cfg(layer, l1):
    return VisionModelSAERunnerConfig(
        model_name=MODEL_ID,
        model_class_name='HookedViT',
        hook_point_layer=layer,
        layer_subtype=HOOK_POINT,
        d_in=768,
        expansion_factor=4,
        context_size=196,
        image_size=224,
        activation_fn_str='relu',
        num_epochs=0.004,                # short, just for tuning
        l1_coefficient=l1,
        lr=LR,
        train_batch_size=4096,
        n_checkpoints=0,
        log_to_wandb=False,   
        checkpoint_path=SAVE_DIR,
        verbose=False,                   # quieter for a sweep
        _device=device,
    )


LAYER = 8 
for l1 in [1e-3, 5e-3, 1e-2, 2e-2]:
    cfg = make_cfg(LAYER, l1)
    trainer = VisionSAETrainer(cfg, model, train_ds, eval_dataset=eval_ds)
    sae = trainer.run()
    print(f'>>> l1={l1}: check the L0 / MSE in the progress bar above')

Not saving checkpoints so skipping creating checkpoint directory


Objective value: 1362368.6250:   2%|▏         | 4/200 [00:00<00:02, 70.14it/s]
Training SAE: Loss: 0.2276, MSE Loss: 0.0187, L1 Loss: 0.2089, L0: 824.3130: : 1019904it [02:25, 7010.77it/s]                           

>>> l1=0.001: check the L0 / MSE in the progress bar above


Not saving checkpoints so skipping creating checkpoint directory


Objective value: 1373633.2500:   1%|          | 2/200 [00:00<00:03, 51.17it/s]
Training SAE: Loss: 1.0760, MSE Loss: 0.0187, L1 Loss: 1.0573, L0: 828.6958: : 1019904it [02:25, 7010.76it/s]                           

...

Objective value: 1360509.2500:   2%|▏         | 3/200 [00:00<00:03, 60.78it/s]
Training SAE: Loss: 1.8509, MSE Loss: 0.0184, L1 Loss: 1.8325, L0: 753.4036: : 1019904it [02:25, 7014.89it/s]                           

>>> l1=0.01: check the L0 / MSE in the progress bar above


Not saving checkpoints so skipping creating checkpoint directory


Objective value: 1364184.7500:   1%|          | 2/200 [00:00<00:03, 51.33it/s]
Training SAE: Loss: 3.6878, MSE Loss: 0.0184, L1 Loss: 3.6694, L0: 754.9792: : 1019904it [02:25, 7018.62it/s]                             

>>> l1=0.02: check the L0 / MSE in the progress bar above


### Full training loop

In [2]:
import os
from google.colab import auth
auth.authenticate_user()
import google.auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload

PARENT_ID = '16axcxIfAwW0ebpc_0NuJSeSy_RDY_zqt'
_creds, _ = google.auth.default()
_drive = build('drive', 'v3', credentials=_creds)

L1_TAG = f'topk_{K}'
FOLDER_NAME = f'sae_siglip_ckpt_{L1_TAG}'

q = (f"name='{FOLDER_NAME}' and '{PARENT_ID}' in parents "
    "and mimeType='application/vnd.google-apps.folder' and trashed=false")
hits = _drive.files().list(q=q, fields='files(id)').execute()['files']
if hits:
    SAE_FOLDER_ID = hits[0]['id']
else:
    meta = {'name': FOLDER_NAME, 'parents': [PARENT_ID],
            'mimeType': 'application/vnd.google-apps.folder'}
    SAE_FOLDER_ID = _drive.files().create(body=meta, fields='id').execute()['id']
print(FOLDER_NAME, 'folder id:', SAE_FOLDER_ID)

def upload_to_drive(path):
    meta  = {'name': os.path.basename(path), 'parents': [SAE_FOLDER_ID]}
    media = MediaFileUpload(path, resumable=True)
    _drive.files().create(body=meta, media_body=media, fields='id').execute()
    print('Uploaded', os.path.basename(path))

MessageError: User cancelled auth_user_ephemeral authorization

In [ ]:
def make_cfg(layer, l1, passes): 
    cfg = VisionModelSAERunnerConfig(
        model_name=MODEL_ID,
        model_class_name='HookedViT',
        hook_point_layer=layer,
        layer_subtype=HOOK_POINT,
        d_in=768,
        expansion_factor=4,
        context_size=196,
        activation_fn_str='topk',
        activation_fn_kwargs={'k': 32}, 
        image_size=224,
        activation_fn_str='relu',
        l1_coefficient=l1,
        lr=LR,
        train_batch_size=4096,
        n_checkpoints=0,
        log_to_wandb=False,
        checkpoint_path=SAVE_DIR,
        verbose=False, 
        _device=device,
    )
    # library hardcodes dataset_size=1.3M; scale so num_epochs = `passes` over our real data
    cfg.num_epochs = passes * len(train_ds) / 1_300_000
    return cfg

HOOK_POINT = 'hook_resid_post'
L1 = 0.0
K = 32
PASSES = 3

for LAYER in range(12):
    cfg = make_cfg(LAYER, L1, PASSES)
    trainer = VisionSAETrainer(cfg, model, train_ds, eval_dataset=eval_ds)
    sae = trainer.run()
    save_path = f'{SAVE_DIR}/sae_{MODEL_ID.split("/")[-1]}_{HOOK_POINT}_{L1_TAG}_layer{LAYER}.pt'
    torch.save(sae.state_dict(), save_path)
    upload_to_drive(save_path)
    print(f'=== layer {LAYER} done, L0/MSE in bar above ===')

: 